# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a guided approach for loading and exploring the [FAIR^2](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the Croissant dataset
dataset = mlc.Dataset(croissant_url)

# Access top-level metadata (as an object, not a dict)
print(f"Dataset name: {dataset.metadata.name}\nDescription: {dataset.metadata.description}")
print(f"License: {dataset.metadata.license}")
print(f"Published: {dataset.metadata.datePublished}, Identifier: {dataset.metadata.identifier}")
print(f"Keywords: {', '.join(dataset.metadata.keywords)}")

## 2. Data Overview
Review available record sets, and their fields and columns. All Croissant entities are referenced by their `@id` for clarity and reproducibility.

In [ ]:
# List all available record sets by @id and their fields (with their @id)
record_sets = list(dataset.record_sets)

print(f"There are {len(record_sets)} record set(s) in this dataset.\n")
for rs in record_sets:
    print(f"RecordSet @id: {rs.id}")
    print(f"  Name: {rs.name}")
    print(f"  Description: {rs.description}")
    print(f"  Fields:")
    for f in rs.fields:
        print(f"    - Field @id: {f.id}\n      name: {f.name}\n      data_type: {f.data_type}\n      source_column: {f.source_column}")
    print("")

## 3. Data Extraction
Load data from the desired record set into a DataFrame for analysis. Here, we demonstrate loading and displaying the first few records from one or more principal record sets, with all variables referenced by their `@id`.

In [ ]:
# Gather @ids of all available record sets for extraction
record_set_ids = [rs.id for rs in record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    print(f"Loading records for RecordSet @id: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"  Columns: {df.columns.tolist()}")
    print(f"  Preview:\n{df.head()}\n")
# For convenience, select the first record set as the main one for analysis:
main_record_set_id = record_set_ids[0] if len(record_set_ids) else None

## 4. Exploratory Data Analysis (EDA)
We'll demonstrate the analysis on the main record set. We'll select suitable numeric and grouping fields using their `@id`s. Here you can filter numeric fields, normalize values, and group by categorical keys.

*Replace the example field `@id`s below if you want to analyze a different field or record set.*

In [ ]:
if main_record_set_id and not dataframes[main_record_set_id].empty:
    df = dataframes[main_record_set_id]
    
    # List all columns as guidance (these map to field @id)
    print(f"All record field @ids (columns):\n{df.columns.tolist()}\n")
    
    # Example: choose a numeric field and a group field by @id, adjust as necessary
    # (Below are plausible generic IDs; replace with actual ones as listed above)
    numeric_field_id = df.select_dtypes(include='number').columns[0] if len(df.select_dtypes(include='number').columns) else None
    group_field_id = None
    for c in df.columns:
        if 'group' in c.lower() or 'sample' in c.lower() or 'type' in c.lower():
            group_field_id = c
            break
        
    if numeric_field_id:
        threshold = df[numeric_field_id].mean()  # use mean as example threshold
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {{numeric_field_id}} > {{threshold:.2f}}:")
        print(filtered_df.head())
        
        # Normalize
        filtered_df[f"{numeric_field_id}_normalized"] = (
            filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
        ) / (filtered_df[numeric_field_id].std() + 1e-9)
        print(f"\nNormalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
        
        # Group by a group_field if present
        if group_field_id:
            grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
            print(f"\nGrouped data by {group_field_id}:")
            print(grouped_df.head())
    else:
        print("No numeric fields found for EDA.")
else:
    print("No data loaded to perform EDA.")

## 5. Visualization
Visualize the distribution of a numerical field or the relationship between two fields. Legend and axis labels use the `@id` (column name) for transparency.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id and not dataframes[main_record_set_id].empty and numeric_field_id:
    plt.figure(figsize=(7, 4))
    sns.histplot(dataframes[main_record_set_id][numeric_field_id], kde=True)
    plt.title(f'Distribution of field {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.tight_layout()
    plt.show()
    
    if group_field_id:
        plt.figure(figsize=(8, 5))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=dataframes[main_record_set_id])
        plt.title(f'{numeric_field_id} grouped by {group_field_id}')
        plt.tight_layout()
        plt.show()

## 6. Conclusion
In this notebook, we've loaded and explored the FAIR^2 dataset on mass spectrometry of extracellular vesicles from stimulated human mast cells, using the `mlcroissant` library and referencing all entities by their `@id`. We:

- Loaded and summarized dataset metadata
- Listed record sets and fields by `@id`
- Loaded and previewed primary data tables
- Filtered and normalized numeric fields and explored grouped aggregates
- Visualized data distributions

**Further analysis:** The structure and field `@id` references from the Croissant schema make downstream processing and documentation robust, enabling reproducible FAIR data workflows.